# Polygon API Exploration
Explore bars + financials endpoints before hardening into `PolygonClient`.

In [ ]:
import os
from dotenv import load_dotenv
from polygon import RESTClient
import pandas as pd

load_dotenv()  # loads POLYGON_API_KEY from .env
client = RESTClient(api_key=os.environ['POLYGON_API_KEY'])
print('connected')

## 1. Hourly Bars

In [ ]:
# Fetch 1 month of hourly bars for AAPL
bars = []
for bar in client.list_aggs(
    ticker='AAPL',
    multiplier=1,
    timespan='hour',
    from_='2024-01-01',
    to='2024-02-01',
    adjusted=True,
    sort='asc',
    limit=50000,
):
    bars.append(bar)

print(f'{len(bars)} bars fetched')
print(vars(bars[0]))  # inspect raw fields

In [ ]:
# Convert to DataFrame
df = pd.DataFrame([vars(b) for b in bars])
df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms', utc=True)
df = df.set_index('timestamp')
df.head()

## 2. Financials (book value)

In [ ]:
# Fetch annual financials for AAPL
financials = []
for f in client.vx.list_stock_financials(
    ticker='AAPL',
    timeframe='annual',
    limit=10,
):
    financials.append(f)

print(f'{len(financials)} filings')
print(vars(financials[0]))  # inspect top-level fields

In [ ]:
# Drill into balance sheet — looking for stockholders equity / book value
f0 = financials[0]
print('filing date:', f0.filing_date)
print('period:', f0.start_date, '->', f0.end_date)
print()
bs = f0.financials.balance_sheet
print(vars(bs))

In [ ]:
# Extract equity and shares to compute book value per share
for f in financials:
    bs = f.financials.balance_sheet
    equity = getattr(bs, 'equity', None)
    shares = getattr(bs, 'common_stock_shares_outstanding', None)
    bvps = equity.value / shares.value if equity and shares else None
    print(f.end_date, '| equity:', equity.value if equity else None,
          '| shares:', shares.value if shares else None,
          '| BVPS:', round(bvps, 2) if bvps else None)

## 3. Sanity check B/M for AAPL
B/M = book value per share / price at filing date

In [ ]:
# Spot check: get daily close on the filing date and compute B/M
f0 = financials[0]
bs = f0.financials.balance_sheet
equity = bs.equity.value
shares = bs.common_stock_shares_outstanding.value
bvps = equity / shares

price_bars = list(client.list_aggs(
    ticker='AAPL',
    multiplier=1,
    timespan='day',
    from_=f0.end_date,
    to=f0.end_date,
    adjusted=True,
))
price = price_bars[0].close if price_bars else None

print(f'Period end: {f0.end_date}')
print(f'BVPS: {bvps:.2f}')
print(f'Price: {price}')
print(f'B/M: {bvps / price:.4f}' if price else 'no price')